## Step 1: Install Dependencies

In [ ]:
!pip install yt-dlp ultralytics datasets huggingface_hub
!apt-get install -y ffmpeg

## Step 2: Download Video and Extract Frames

In [ ]:
!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]" \n  -o "input_video.mp4" \n  "https://www.youtube.com/watch?v=YcvECxtXoxQ"

In [ ]:
!mkdir -p frames
!ffmpeg -i input_video.mp4 -vf "fps=1/5" frames/frame_%04d.jpg

## Step 3: Train Car Parts Detector

Fine-tune YOLOv8n-seg on the Ultralytics Car Parts Segmentation dataset (23 classes).

In [ ]:
from ultralytics import YOLO

# Load base model and fine-tune on car parts dataset
model = YOLO("yolov8n-seg.pt")
model.train(data="carparts-seg.yaml", epochs=10, imgsz=640)
print(model.names)

## Step 4: Run Detection on Video Frames and Build Parquet Index

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

# Load best trained weights
model = YOLO("/content/runs/segment/train/weights/best.pt")

frames_dir = "frames"
rows = []

for fname in sorted(os.listdir(frames_dir)):
    if not fname.endswith(".jpg"):
        continue
    frame_index = int(fname.split("_")[1].split(".")[0])
    timestamp_sec = frame_index * 5

    results = model(os.path.join(frames_dir, fname))
    for box in results[0].boxes:
        rows.append({
            "video_id": "YcvECxtXoxQ",
            "frame_index": frame_index,
            "timestamp_sec": timestamp_sec,
            "class_label": model.names[int(box.cls)],
            "confidence_score": float(box.conf),
            "x_min": float(box.xyxy[0][0]),
            "y_min": float(box.xyxy[0][1]),
            "x_max": float(box.xyxy[0][2]),
            "y_max": float(box.xyxy[0][3]),
        })

df = pd.DataFrame(rows)
df.to_parquet("video_detections.parquet", index=False)
print(f"Saved {len(df)} detections")
print(df["class_label"].value_counts())

## Step 5: Image-to-Video Retrieval

Load query images from Hugging Face, detect car parts in each, and match against the video index.

In [ ]:
from datasets import load_dataset

# Load query images
ds = load_dataset("aegean-ai/rav4-exterior-images", split="train")

# Load the video index
df = pd.read_parquet("video_detections.parquet")

results = []

for item in ds:
    img = item["image"]

    # Run detector on query image
    query_results = model(img)

    if len(query_results[0].boxes) == 0:
        continue

    # Get detected classes in query image
    query_classes = set([model.names[int(c)] for c in query_results[0].boxes.cls])

    # Find matching frames in video index
    matches = df[df["class_label"].isin(query_classes)]

    if len(matches) == 0:
        continue

    # Get contiguous time intervals
    timestamps = sorted(matches["timestamp_sec"].unique())
    start = timestamps[0]
    end = timestamps[0]

    for t in timestamps[1:]:
        if t - end <= 10:  # within 10 seconds = same clip
            end = t
        else:
            results.append({
                "query_timestamp": item["timestamp"],
                "query_classes": str(query_classes),
                "start_timestamp": start,
                "end_timestamp": end,
                "number_of_supporting_detections": len(matches)
            })
            start = end = t

    results.append({
        "query_timestamp": item["timestamp"],
        "query_classes": str(query_classes),
        "start_timestamp": start,
        "end_timestamp": end,
        "number_of_supporting_detections": len(matches)
    })

results_df = pd.DataFrame(results)
results_df.to_parquet("retrieval_results.parquet", index=False)
print(f"Saved {len(results_df)} retrieval results")
print(results_df.head(10))

## Step 6: Upload to Hugging Face

In [ ]:
from huggingface_hub import HfApi, notebook_login

notebook_login()

api = HfApi()
api.create_repo(repo_id="ZuhairMunawar/rav4-video-detections", repo_type="dataset", exist_ok=True)

api.upload_file(
    path_or_fileobj="video_detections.parquet",
    path_in_repo="video_detections.parquet",
    repo_id="ZuhairMunawar/rav4-video-detections",
    repo_type="dataset"
)

api.upload_file(
    path_or_fileobj="retrieval_results.parquet",
    path_in_repo="retrieval_results.parquet",
    repo_id="ZuhairMunawar/rav4-video-detections",
    repo_type="dataset"
)

print("Uploaded successfully!")